# GP Kernel Addition Animation

This notebook uses a shared 100-point dataset sampled from a GP with a composite kernel: linear + periodic + squared-exponential.

The animation has four frames:
1. Data only
2. Posterior with linear kernel
3. Posterior with linear + periodic kernel
4. Posterior with linear + periodic + exp-square kernel

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import seaborn as sns

from data_creation import generate_gp_kernel_addition_data
from gp_utils import gp_posterior
from kernels import linear_kernel, periodic_kernel, squared_exponential

sns.set_theme(style="whitegrid", palette="husl")
sns.set_context("notebook", font_scale=1.1)

In [ ]:
X_train, y_train, sn = generate_gp_kernel_addition_data()
X_plot = np.linspace(-5.5, 5.5, 320)
sigma = 2.0


def kernel_linear(x1, x2):
    return linear_kernel(x1, x2, slope_variance=0.25, bias_variance=0.8)


def kernel_linear_periodic(x1, x2):
    return kernel_linear(x1, x2) + periodic_kernel(
        x1, x2, lengthscale=1.0, variance=0.9, period=2.0
    )


def kernel_linear_periodic_exp_square(x1, x2):
    return kernel_linear_periodic(x1, x2) + squared_exponential(
        x1, x2, lengthscale=1.2, variance=0.7
    )


frame_specs = [
    {
        "title": "Frame 1: Data",
        "kernel": None,
        "overlay": None,
    },
    {
        "title": "Frame 2: Posterior (Linear)",
        "kernel": kernel_linear,
        "overlay": "k(x, x') = linear",
    },
    {
        "title": "Frame 3: Posterior (Linear + Periodic)",
        "kernel": kernel_linear_periodic,
        "overlay": "k(x, x') = linear + periodic",
    },
    {
        "title": "Frame 4: Posterior (Linear + Periodic + Exp-Square)",
        "kernel": kernel_linear_periodic_exp_square,
        "overlay": "k(x, x') = linear + periodic + exp-square",
    },
]

frames = []
y_min = np.min(y_train - sn)
y_max = np.max(y_train + sn)

for spec in frame_specs:
    frame = dict(spec)
    if spec["kernel"] is not None:
        mu, cov = gp_posterior(X_train, y_train, X_plot, spec["kernel"], sn)
        sd = np.sqrt(np.clip(np.diag(cov), 0.0, None))
        frame["mu"] = mu
        frame["sd"] = sd
        y_min = min(y_min, np.min(mu - sigma * sd))
        y_max = max(y_max, np.max(mu + sigma * sd))
    frames.append(frame)

margin = 0.1 * (y_max - y_min)
y_min -= margin
y_max += margin

print(f"Prepared {len(frames)} animation frames using {len(X_train)} data points.")

In [ ]:
prop_cycle = plt.rcParams["axes.prop_cycle"]
colors = prop_cycle.by_key()["color"]
ci_color = colors[0]

fig, ax = plt.subplots(figsize=(10, 6))


def draw_frame(frame_idx):
    frame = frames[frame_idx]
    ax.cla()

    if frame["kernel"] is not None:
        mu = frame["mu"]
        sd = frame["sd"]
        ax.fill_between(
            X_plot,
            mu - sigma * sd,
            mu + sigma * sd,
            color=ci_color,
            alpha=0.35,
            label="95% confidence interval",
        )
        ax.plot(X_plot, mu, color=ci_color, lw=2.2, label="Posterior mean")

    ax.errorbar(
        X_train, y_train, yerr=sn,
        fmt="o", ms=5, elinewidth=1.5, capsize=3,
        label="Observed data", color="steelblue", alpha=0.85,
    )

    if frame["overlay"] is not None:
        ax.text(
            0.03,
            0.96,
            frame["overlay"],
            transform=ax.transAxes,
            fontsize=12,
            verticalalignment="top",
            bbox={"boxstyle": "round", "facecolor": "white", "alpha": 0.85},
        )

    ax.set_title(frame["title"], fontsize=14, fontweight="bold")
    ax.set_xlabel("Input (x)", fontsize=12)
    ax.set_ylabel("Output (y)", fontsize=12)
    ax.set_xlim(X_plot[0], X_plot[-1])
    ax.set_ylim(y_min, y_max)
    ax.legend(loc="best", frameon=True, shadow=True)
    ax.grid(True, alpha=0.3)


anim = animation.FuncAnimation(
    fig,
    draw_frame,
    frames=len(frames),
    interval=1700,
    repeat=True,
)

plt.tight_layout()
plt.close(fig)
HTML(anim.to_jshtml())